# ProxyFace · Emotion Classifier v8 — bert-tiny, fast & accurate

**Back to bert-tiny (4MB INT8, <100ms WebGPU)** — properly trained this time.

Root cause of v6/v7 failures:
- v6: `Counter` not imported in training cell → class weights silently broken → uniform loss → poor generalization
- v7: switched to bert-small (27MB, 900ms) which is too slow for real-time use

v8 fixes:
- Every cell is fully self-contained (no cross-cell import dependencies)
- Split data FIRST, augment TRAIN set only → no validation data leakage
- 40 epochs, LR=2e-5, no label smoothing (tiny models need clean gradients)
- Synonym phrases injected into train set only
- ~3200 base rows → ~7000 after augmentation

## 1. GPU

In [1]:
!nvidia-smi --query-gpu=name,memory.total --format=csv,noheader
import torch
assert torch.cuda.is_available(), "No GPU — check Runtime > Change runtime type > T4"
print(f"GPU: {torch.cuda.get_device_name(0)}")
print(f"Memory: {torch.cuda.get_device_properties(0).total_memory/1e9:.1f} GB")

Tesla T4, 15360 MiB
GPU: Tesla T4
Memory: 15.6 GB


## 2. Install

In [2]:
!pip install -q "numpy==2.0.2" --force-reinstall
!pip install -q \
    "transformers==4.45.2" \
    "tokenizers>=0.20,<0.21" \
    "onnx>=1.17,<1.20" \
    "onnxruntime>=1.19,<1.21" \
    "onnxscript>=0.1.0" \
    "scikit-learn>=1.5,<2.0"

import torch, transformers, onnx, onnxruntime
print(f"torch        {torch.__version__}")
print(f"transformers {transformers.__version__}")
print(f"onnx         {onnx.__version__}")
print(f"onnxruntime  {onnxruntime.__version__}")
print(f"CUDA         {torch.cuda.is_available()}")

     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 60.9/60.9 kB 3.2 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 19.2/19.2 MB 95.6 MB/s eta 0:00:00
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 44.4/44.4 kB 3.2 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 9.9/9.9 MB 96.4 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 3.0/3.0 MB 130.1 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 18.2/18.2 MB 86.9 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 13.3/13.3 MB 111.5 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 714.8/714.8 kB 58.2 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 566.4/566.4 kB 47.2 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 166.8/166.8 kB 20.2 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 46.0/46.0 kB 5.0 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 86.8/86.8 kB 10.3 MB/s eta 0:00:00
torch        2.10.0+cu128
transformers 4.45.2
o

## 3. Config

In [3]:
from pathlib import Path

WORK_DIR = Path('/content/pf8')
CKPT_DIR = WORK_DIR / 'ckpt'
OUT_DIR  = WORK_DIR / 'out'
for d in [WORK_DIR, CKPT_DIR, OUT_DIR]:
    d.mkdir(parents=True, exist_ok=True)

# Labels — DO NOT REORDER (must match TypeScript EMOTIONS array)
LABELS   = ['IDLE','THINKING','HAPPY','SAD','ANGRY','SURPRISED','EXPLAINING','ERROR']
LABEL2ID = {l: i for i, l in enumerate(LABELS)}

# bert-tiny: 4.4M params → 4MB INT8 → <100ms WebGPU latency
MODEL_BASE   = 'prajjwal1/bert-tiny'
MAX_LENGTH   = 96
BATCH_SIZE   = 32
EPOCHS       = 40
LR           = 2e-5     # lower than v7 — tiny model needs gentle updates
WARMUP_RATIO = 0.1
WEIGHT_DECAY = 0.01
LABEL_SMOOTH = 0.0      # ZERO — tiny model needs clean hard targets
SEED         = 42

print("Config:")
print(f"  model:      {MODEL_BASE}")
print(f"  epochs:     {EPOCHS}")
print(f"  lr:         {LR}")
print(f"  label_smooth: {LABEL_SMOOTH} (off — critical for tiny model)")

Config:
  model:      prajjwal1/bert-tiny
  epochs:     40
  lr:         2e-05
  label_smooth: 0.0 (off — critical for tiny model)


## 4. Load dataset + split + augment train only

Upload `proxyface_emotions.jsonl` via Files sidebar.

**Split happens BEFORE augmentation** so val/test sets are clean originals.

In [5]:
import json, random
from pathlib import Path
from collections import Counter
from sklearn.model_selection import train_test_split

random.seed(SEED)

# ── Load ──────────────────────────────────────────────────────────────────
candidates = list(Path('/content').glob('proxyface_emotions*.jsonl'))
if not candidates:
    raise SystemExit('Upload proxyface_emotions*.jsonl via the Files sidebar first')

src = max(candidates, key=lambda p: p.stat().st_mtime)
print(f"Source: {src}")

base_rows = []
with open(src, encoding='utf-8') as f:
    for i, line in enumerate(f, 1):
        line = line.strip()
        if not line: continue
        try:
            obj = json.loads(line)
            label = obj.get('label') or LABELS[int(obj['label_id'])]
            text  = str(obj['text']).strip()
            if label in LABEL2ID and len(text) >= 4:
                base_rows.append({'text': text, 'label': label})
        except Exception as e:
            if i <= 3: print(f"  skip line {i}: {e}")

print(f"Base rows: {len(base_rows)}")

# ── Split FIRST (clean val/test, no augmentation contamination) ───────────
all_texts  = [r['text']  for r in base_rows]
all_labels = [LABEL2ID[r['label']] for r in base_rows]

tr_t, tmp_t, tr_l, tmp_l = train_test_split(
    all_texts, all_labels, test_size=0.20, stratify=all_labels, random_state=SEED)
va_t, te_t, va_l, te_l = train_test_split(
    tmp_t, tmp_l, test_size=0.50, stratify=tmp_l, random_state=SEED)
print(f"Split: train={len(tr_t)} val={len(va_t)} test={len(te_t)}")

# ── Augment TRAIN set only ────────────────────────────────────────────────
# Class-specific prefixes reinforce the emotion signal for the tiny model
PREFIXES = {
    'IDLE':       ['Sure, ','Got it. ','Understood. ','Okay. ','Noted. '],
    'THINKING':   ['Hmm, ','Let me think... ','Well, ','Actually, ','Hmm... '],
    'HAPPY':      ['Great news: ','Excellent! ','Yes! ','Wonderful! ',''],
    'SAD':        ['Unfortunately, ','Sadly, ','Regrettably, ','sorry, '],
    'ANGRY':      ['Absolutely not. ','No. ','This is unacceptable: ','I refuse: '],
    'SURPRISED':  ['Wow! ','Oh! ','Wait — ','Incredible! ','I cannot believe it: '],
    'EXPLAINING': ['To summarize: ','Here is how: ','Note that ','In short: '],
    'ERROR':      ['[ERROR] ','FAILED: ','Exception: ','Warning: ','Critical: '],
}

# Synonym PHRASES (not bare words — BERT needs context)
SYN_PHRASES = {
    'IDLE':       ['Everything seems okay.','That is fine.','Noted, understood.',
                   'All is normal.','Alright, confirmed.','Sure, acknowledged.',
                   'Received and understood.','Everything is stable.',
                   'That is routine.','Calm and steady.'],
    'THINKING':   ['Hmm, let me think.','I am considering this.',
                   'Let me ponder that.','I am deliberating.',
                   'Still processing this.','Let me evaluate carefully.',
                   'I am weighing the options.','Hmm, analyzing now.',
                   'I am reflecting on that.','Let me reason through this.'],
    'HAPPY':      ['That is wonderful!','I feel great about this!',
                   'This is excellent news!','I am so pleased!',
                   'Fantastic, I am thrilled!','This is simply amazing!',
                   'I feel elated!','How delightful!',
                   'I am overjoyed!','That is brilliant!'],
    'SAD':        ['I feel such sorrow.','Such grief fills me.',
                   'A deep despair overcomes me.','I feel so melancholy.',
                   'My heart is broken.','I am devastated.',
                   'Such sadness overwhelms me.','I feel hopeless.',
                   'A terrible regret fills me.','I am so sorry for this loss.'],
    'ANGRY':      ['I am furious about this!','How livid I am!',
                   'I am outraged!','This is absolutely intolerable!',
                   'I am incensed!','How infuriating!',
                   'I am enraged!','This is completely unacceptable!',
                   'I am seething with anger.','How indignant I feel!'],
    'SURPRISED':  ['Wow, unbelievable!','That is astonishing!',
                   'I am truly astounded!','How shocking!',
                   'That is absolutely stunning!','Incredible, I had no idea!',
                   'Whoa, that is remarkable!','I am completely stunned!',
                   'How extraordinary!','That is mind-blowing!'],
    'EXPLAINING': ['Firstly, let me explain.','Secondly, as I noted.',
                   'Therefore, to clarify:','Furthermore, I should add.',
                   'To elaborate on that point:','Step by step, here is how.',
                   'In other words, to be specific:','To summarize the key steps:',
                   'Consequently, the result is:','In particular, note that:'],
    'ERROR':      ['An error occurred.','A critical failure was detected.',
                   'System crash: exception raised.','TypeError was thrown.',
                   'A fatal error has occurred.','Request timeout: connection refused.',
                   'NaN detected in output.','Access denied: unauthorized.',
                   'Traceback: null reference.','Invalid input: malformed data.'],
}

aug_train_t, aug_train_l = list(tr_t), list(tr_l)

# Add prefix variants to train set
for text, label_id in zip(tr_t, tr_l):
    label = LABELS[label_id]
    pfxs  = [p for p in PREFIXES[label] if p]
    if pfxs:
        aug_train_t.append(random.choice(pfxs) + text)
        aug_train_l.append(label_id)

# Add synonym phrases to train set (10 per class = 80 total)
for label, phrases in SYN_PHRASES.items():
    lid = LABEL2ID[label]
    for phrase in phrases:
        aug_train_t.append(phrase)
        aug_train_l.append(lid)

# Shuffle train
combined = list(zip(aug_train_t, aug_train_l))
random.shuffle(combined)
aug_train_t, aug_train_l = zip(*combined)

print(f"After augmentation: train={len(aug_train_t)} val={len(va_t)} test={len(te_t)}")
print("Train class distribution:")
cnt = Counter(aug_train_l)
for i, label in enumerate(LABELS):
    bar = '█' * (cnt[i] // 10)
    print(f"  {label:12s} {cnt[i]:>4d}  {bar}")

Source: /content/proxyface_emotions.jsonl
Base rows: 3200
Split: train=2560 val=320 test=320
After augmentation: train=5200 val=320 test=320
Train class distribution:
  IDLE          650  █████████████████████████████████████████████████████████████████
  THINKING      650  █████████████████████████████████████████████████████████████████
  HAPPY         650  █████████████████████████████████████████████████████████████████
  SAD           650  █████████████████████████████████████████████████████████████████
  ANGRY         650  █████████████████████████████████████████████████████████████████
  SURPRISED     650  █████████████████████████████████████████████████████████████████
  EXPLAINING    650  █████████████████████████████████████████████████████████████████
  ERROR         650  █████████████████████████████████████████████████████████████████


## 5. Train — bert-tiny, 40 epochs, cosine LR, class weights

In [6]:
# All imports self-contained in this cell
import random, json
from collections import Counter
from pathlib import Path

import numpy as np
import torch
import torch.nn as nn
from torch.optim import AdamW
from torch.utils.data import Dataset, DataLoader
from sklearn.metrics import f1_score, classification_report
from transformers import (
    AutoTokenizer, AutoModelForSequenceClassification,
    get_cosine_schedule_with_warmup,
)

random.seed(SEED); np.random.seed(SEED); torch.manual_seed(SEED)
if torch.cuda.is_available(): torch.cuda.manual_seed_all(SEED)

DEVICE = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print(f"Device: {DEVICE}")

tokenizer = AutoTokenizer.from_pretrained(MODEL_BASE)

class EmotionDS(Dataset):
    def __init__(self, texts, labels):
        enc = tokenizer(list(texts), truncation=True, padding='max_length',
                        max_length=MAX_LENGTH, return_tensors='pt')
        self.ids   = enc['input_ids']
        self.mask  = enc['attention_mask']
        self.ttype = enc.get('token_type_ids', torch.zeros_like(enc['input_ids']))
        self.lbls  = torch.tensor(list(labels), dtype=torch.long)
    def __len__(self): return len(self.lbls)
    def __getitem__(self, i):
        return self.ids[i], self.mask[i], self.ttype[i], self.lbls[i]

tr_ds = EmotionDS(aug_train_t, aug_train_l)
va_ds = EmotionDS(va_t, va_l)
te_ds = EmotionDS(te_t, te_l)

tr_dl = DataLoader(tr_ds, batch_size=BATCH_SIZE, shuffle=True,  num_workers=2, pin_memory=True)
va_dl = DataLoader(va_ds, batch_size=64,         shuffle=False, num_workers=2, pin_memory=True)
te_dl = DataLoader(te_ds, batch_size=64,         shuffle=False, num_workers=2, pin_memory=True)

model = AutoModelForSequenceClassification.from_pretrained(
    MODEL_BASE, num_labels=len(LABELS),
    id2label={i: l for i, l in enumerate(LABELS)},
    label2id=LABEL2ID, ignore_mismatched_sizes=True,
).to(DEVICE)
print(f"Params: {sum(p.numel() for p in model.parameters()):,}")

# Class weights — computed from AUGMENTED train set
cnt      = Counter(aug_train_l)
n_total  = len(aug_train_l)
weights  = torch.tensor(
    [n_total / (len(LABELS) * cnt[i]) for i in range(len(LABELS))],
    dtype=torch.float32, device=DEVICE)
print(f"Weights: {[f'{w:.2f}' for w in weights.tolist()]}")

# label_smoothing=0.0 — tiny model needs clean hard targets
criterion = nn.CrossEntropyLoss(weight=weights, label_smoothing=LABEL_SMOOTH)
optimizer = AdamW(model.parameters(), lr=LR, weight_decay=WEIGHT_DECAY)

total_steps  = len(tr_dl) * EPOCHS
warmup_steps = int(total_steps * WARMUP_RATIO)
scheduler    = get_cosine_schedule_with_warmup(
    optimizer, num_warmup_steps=warmup_steps, num_training_steps=total_steps)
print(f"Steps: {total_steps}  Warmup: {warmup_steps}")

def evaluate(dl):
    model.eval()
    preds, gold = [], []
    with torch.no_grad():
        for ids, mask, ttype, lbl in dl:
            out = model(input_ids=ids.to(DEVICE),
                        attention_mask=mask.to(DEVICE),
                        token_type_ids=ttype.to(DEVICE))
            preds.extend(out.logits.argmax(-1).cpu().tolist())
            gold.extend(lbl.tolist())
    acc = sum(p == g for p, g in zip(preds, gold)) / len(gold)
    f1  = f1_score(gold, preds, average='macro', zero_division=0)
    return acc, f1, preds, gold

best_f1, best_epoch = 0.0, 0
print(f"\n{'Ep':>3}  {'Loss':>8}  {'Val-Acc':>7}  {'Val-F1':>6}")
print("─" * 35)

for epoch in range(1, EPOCHS + 1):
    model.train()
    total_loss, steps = 0.0, 0
    for ids, mask, ttype, lbl in tr_dl:
        ids, mask, ttype, lbl = (ids.to(DEVICE), mask.to(DEVICE),
                                  ttype.to(DEVICE), lbl.to(DEVICE))
        optimizer.zero_grad()
        loss = criterion(
            model(input_ids=ids, attention_mask=mask,
                  token_type_ids=ttype).logits, lbl)
        loss.backward()
        nn.utils.clip_grad_norm_(model.parameters(), 1.0)
        optimizer.step()
        scheduler.step()
        total_loss += loss.item(); steps += 1

    va_acc, va_f1, _, _ = evaluate(va_dl)
    mark = " ◀" if va_f1 > best_f1 else ""
    print(f"{epoch:>3}  {total_loss/steps:>8.4f}  {va_acc:>7.4f}  {va_f1:>6.4f}{mark}")

    if va_f1 > best_f1:
        best_f1, best_epoch = va_f1, epoch
        model.save_pretrained(str(CKPT_DIR))
        tokenizer.save_pretrained(str(CKPT_DIR))

print(f"\nBest epoch {best_epoch}  val F1 {best_f1:.4f}")

# Test on best checkpoint
model = AutoModelForSequenceClassification.from_pretrained(str(CKPT_DIR)).to(DEVICE)
te_acc, te_f1, te_preds, te_gold = evaluate(te_dl)
print(f"Test accuracy {te_acc:.4f}   Test macro-F1 {te_f1:.4f}")
print()
print(classification_report(te_gold, te_preds, target_names=LABELS, digits=3))

Device: cuda


/usr/local/lib/python3.12/dist-packages/huggingface_hub/utils/_auth.py:94: UserWarning: 
The secret `HF_TOKEN` does not exist in your Colab secrets.
To authenticate with the Hugging Face Hub, create a token in your settings tab (https://huggingface.co/settings/tokens), set it as secret in your Google Colab and restart your session.
You will be able to reuse this secret in all of your notebooks.
Please note that authentication is recommended but still optional to access public models or datasets.
  warnings.warn(


config.json:   0%|          | 0.00/285 [00:00<?, ?B/s]

vocab.txt: 0.00B [00:00, ?B/s]

pytorch_model.bin:   0%|          | 0.00/17.8M [00:00<?, ?B/s]

Some weights of BertForSequenceClassification were not initialized from the model checkpoint at prajjwal1/bert-tiny and are newly initialized: ['classifier.bias', 'classifier.weight']
You should probably TRAIN this model on a down-stream task to be able to use it for predictions and inference.


Params: 4,386,952
Weights: ['1.00', '1.00', '1.00', '1.00', '1.00', '1.00', '1.00', '1.00']
Steps: 6520  Warmup: 652

 Ep      Loss  Val-Acc  Val-F1
───────────────────────────────────
  1    2.0796   0.2062  0.1161 ◀
  2    2.0348   0.3563  0.3109 ◀
  3    1.9329   0.5031  0.4632 ◀
  4    1.7606   0.6188  0.5968 ◀
  5    1.5338   0.6719  0.6594 ◀
  6    1.3123   0.7281  0.7171 ◀
  7    1.1247   0.7500  0.7370 ◀
  8    0.9690   0.7719  0.7604 ◀
  9    0.8407   0.7875  0.7765 ◀
 10    0.7329   0.7937  0.7839 ◀
 11    0.6355   0.8094  0.8033 ◀
 12    0.5537   0.8469  0.8414 ◀
 13    0.4861   0.8438  0.8400
 14    0.4304   0.8562  0.8509 ◀
 15    0.3873   0.8438  0.8394
 16    0.3394   0.8594  0.8551 ◀
 17    0.3027   0.8625  0.8590 ◀
 18    0.2708   0.8500  0.8431
 19    0.2517   0.8594  0.8567
 20    0.2284   0.8594  0.8565
 21    0.2110   0.8562  0.8519
 22    0.1958   0.8688  0.8658 ◀
 23    0.1797   0.8500  0.8447
 24    0.1718   0.8656  0.8614
 25    0.1585   0.8594  0.8548
 26    0

## 6. Export ONNX + INT8  (opset 17, dynamo=False)

In [7]:
# All imports self-contained
import json
import torch
import onnx
from onnxruntime.quantization import quantize_dynamic, QuantType
from transformers import AutoModelForSequenceClassification, AutoTokenizer
from pathlib import Path

print("Loading best checkpoint...")
model_exp = AutoModelForSequenceClassification.from_pretrained(str(CKPT_DIR)).cpu()
tok_exp   = AutoTokenizer.from_pretrained(str(CKPT_DIR))
model_exp.eval()

dummy_enc   = tok_exp("hello world", truncation=True, padding='max_length',
                       max_length=MAX_LENGTH, return_tensors='pt')
dummy_ids   = dummy_enc['input_ids']
dummy_mask  = dummy_enc['attention_mask']
dummy_ttype = dummy_enc.get('token_type_ids', torch.zeros_like(dummy_ids))

fp32_path = OUT_DIR / 'model.onnx'
int8_path = OUT_DIR / 'model_int8.onnx'   # exact name browser worker expects

print("Exporting FP32 ONNX...")
with torch.no_grad():
    torch.onnx.export(
        model_exp,
        (dummy_ids, dummy_mask, dummy_ttype),
        str(fp32_path),
        opset_version=17,
        do_constant_folding=True,
        dynamo=False,
        input_names=['input_ids', 'attention_mask', 'token_type_ids'],
        output_names=['logits'],
    )
fp32_mb = fp32_path.stat().st_size / 1e6
print(f"FP32: {fp32_mb:.2f} MB")
onnx.checker.check_model(str(fp32_path))
print("Validation: ✓")

print("Quantizing INT8...")
quantize_dynamic(str(fp32_path), str(int8_path), weight_type=QuantType.QInt8)
int8_mb = int8_path.stat().st_size / 1e6
print(f"INT8: {int8_mb:.2f} MB  ({100*(1-int8_mb/fp32_mb):.0f}% reduction)")
fp32_path.unlink()

tok_exp.save_pretrained(str(OUT_DIR))
(OUT_DIR / 'labels.json').write_text(json.dumps({
    'labels': LABELS,
    'id2label': {str(i): l for i, l in enumerate(LABELS)},
    'version': '0.3.0',
    'model_base': MODEL_BASE,
    'quantization': 'dynamic_int8',
    'opset': 17,
}, indent=2))

print("\nOutput files:")
for f in sorted(OUT_DIR.iterdir()):
    if f.is_file():
        print(f"  {f.name:<35} {f.stat().st_size/1e3:>8.1f} KB")

Loading best checkpoint...
Exporting FP32 ONNX...


/tmp/ipykernel_8901/3292845102.py:25: DeprecationWarning: You are using the legacy TorchScript-based ONNX export. Starting in PyTorch 2.9, the new torch.export-based ONNX exporter has become the default. Learn more about the new export logic: https://docs.pytorch.org/docs/stable/onnx_export.html. For exporting control flow: https://pytorch.org/tutorials/beginner/onnx/export_control_flow_model_to_onnx_tutorial.html
  torch.onnx.export(


FP32: 17.58 MB
Validation: ✓
Quantizing INT8...
INT8: 4.46 MB  (75% reduction)

Output files:
  labels.json                              0.4 KB
  model_int8.onnx                       4456.1 KB
  special_tokens_map.json                  0.7 KB
  tokenizer.json                         711.7 KB
  tokenizer_config.json                    1.5 KB
  vocab.txt                              231.5 KB


## 7. Verify

In [8]:
# All imports self-contained
import numpy as np
import onnxruntime as ort
from transformers import AutoTokenizer
from pathlib import Path

sess  = ort.InferenceSession(str(OUT_DIR / 'model_int8.onnx'),
                              providers=['CPUExecutionProvider'])
tok_v = AutoTokenizer.from_pretrained(str(OUT_DIR))
input_names = {inp.name for inp in sess.get_inputs()}

def predict(text):
    enc  = tok_v(text, truncation=True, padding='max_length',
                  max_length=MAX_LENGTH, return_tensors='np')
    feed = {'input_ids':      enc['input_ids'].astype(np.int64),
            'attention_mask': enc['attention_mask'].astype(np.int64)}
    if 'token_type_ids' in input_names:
        feed['token_type_ids'] = np.zeros_like(enc['input_ids'], dtype=np.int64)
    logits = sess.run(None, feed)[0][0]
    e = np.exp(logits - logits.max()); probs = e / e.sum()
    idx = int(probs.argmax())
    return LABELS[idx], float(probs[idx])

PROBES = [
    # Sentence probes
    ("The river flows from the mountains to the sea.", "IDLE"),
    ("Let me check that document for you.", "IDLE"),
    ("Hmm, give me a moment to think this through.", "THINKING"),
    ("I'm weighing a few options right now.", "THINKING"),
    ("Your changes are saved and the build passed!", "HAPPY"),
    ("We reached the summit just as the clouds parted.", "HAPPY"),
    ("I apologize, I'm not able to access that information.", "SAD"),
    ("The first snowman stood alone until he melted.", "SAD"),
    ("No, I will not assist with that under any circumstances.", "ANGRY"),
    ("That was a textbook hostile takeover of the agenda.", "ANGRY"),
    ("Oh wow, I didn't realize the dataset was that large!", "SURPRISED"),
    ("The old elevator opened onto a floor not on any plan.", "SURPRISED"),
    ("Step one: initialize. Step two: iterate. Step three: return.", "EXPLAINING"),
    ("The observer effect states measurement influences the phenomenon.", "EXPLAINING"),
    ("TypeError: Cannot read property 'foo' of undefined at line 42.", "ERROR"),
    ("I can't clear the error until the root cause is addressed.", "ERROR"),
    # Synonym phrase probes
    ("I feel such sorrow.", "SAD"),
    ("Such grief fills me.", "SAD"),
    ("I am furious about this!", "ANGRY"),
    ("How livid I am!", "ANGRY"),
    ("That is wonderful!", "HAPPY"),
    ("I feel elated!", "HAPPY"),
    ("Hmm, let me think.", "THINKING"),
    ("I am still processing this.", "THINKING"),
    ("An error occurred.", "ERROR"),
    ("Traceback: null reference.", "ERROR"),
    ("Firstly, let me explain.", "EXPLAINING"),
    ("Therefore, to clarify:", "EXPLAINING"),
    ("Wow, unbelievable!", "SURPRISED"),
    ("That is astonishing!", "SURPRISED"),
    ("Everything seems okay.", "IDLE"),
    ("Noted, understood.", "IDLE"),
]

correct = 0
print(f"{'':2} {'Expected':12} {'Pred':12} {'Conf':5}  Text")
print("─" * 72)
for text, expected in PROBES:
    pred, conf = predict(text)
    ok = pred == expected
    correct += int(ok)
    print(f" {'✓' if ok else '✗'} {expected:12} {pred:12} {conf:.2f}  {text[:40]}")

pa = correct / len(PROBES)
n_sent = 16; n_syn = 16
sent_ok = sum(1 for t,e in PROBES[:n_sent] if predict(t)[0]==e)
syn_ok  = sum(1 for t,e in PROBES[n_sent:] if predict(t)[0]==e)
print(f"\nTotal:     {correct}/{len(PROBES)} ({pa:.0%})")
print(f"Sentences: {sent_ok}/{n_sent}")
print(f"Synonyms:  {syn_ok}/{n_syn}")

if pa >= 0.88:   print("\n✓ Ready to ship — excellent quality")
elif pa >= 0.75: print("\n⚠ Acceptable — add more diverse rows and retrain")
else:            print("\n✗ Check val F1 in training — should be >0.82")

   Expected     Pred         Conf   Text
────────────────────────────────────────────────────────────────────────
 ✓ IDLE         IDLE         0.74  The river flows from the mountains to th
 ✓ IDLE         IDLE         0.91  Let me check that document for you.
 ✓ THINKING     THINKING     0.93  Hmm, give me a moment to think this thro
 ✓ THINKING     THINKING     0.47  I'm weighing a few options right now.
 ✗ HAPPY        ANGRY        0.27  Your changes are saved and the build pas
 ✓ HAPPY        HAPPY        0.85  We reached the summit just as the clouds
 ✗ SAD          ANGRY        0.86  I apologize, I'm not able to access that
 ✓ SAD          SAD          0.61  The first snowman stood alone until he m
 ✓ ANGRY        ANGRY        0.90  No, I will not assist with that under an
 ✓ ANGRY        ANGRY        0.45  That was a textbook hostile takeover of 
 ✓ SURPRISED    SURPRISED    0.85  Oh wow, I didn't realize the dataset was
 ✗ SURPRISED    SAD          0.83  The old elevator opened

## 8. Download

Unzip → drop contents into:
```
packages\proxyface-core\src\assets\models\emotion\
```
Directly in `emotion\` — **NOT** in `emotion\onnx\`.

Expected file: `emotion\model_int8.onnx`

Then just `pnpm dev` — no sync needed for model changes.

In [9]:
import shutil
from pathlib import Path

zip_path = '/content/proxyface-emotion-v8.zip'
shutil.make_archive(zip_path[:-4], 'zip', str(OUT_DIR))
mb = Path(zip_path).stat().st_size / 1e6
print(f"Zip: {zip_path}  ({mb:.2f} MB)")
print("\nFiles in zip:")
for f in sorted(OUT_DIR.iterdir()):
    if f.is_file():
        kb = f.stat().st_size / 1e3
        print(f"  {f.name:<35} {kb:>7.1f} KB")

try:
    from google.colab import files
    files.download(zip_path)
    print("\nDownload started ✓")
except ImportError:
    print("Not in Colab — find zip at /content/proxyface-emotion-v8.zip")

Zip: /content/proxyface-emotion-v8.zip  (3.55 MB)

Files in zip:
  labels.json                             0.4 KB
  model_int8.onnx                      4456.1 KB
  special_tokens_map.json                 0.7 KB
  tokenizer.json                        711.7 KB
  tokenizer_config.json                   1.5 KB
  vocab.txt                             231.5 KB


<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>


Download started ✓
